# Lab 13 · seaborn, bản đồ mật độ & phê bình một hình cherry-pick

**Lập trình xử lý dữ liệu (LTXLDL) · 2627-1 · Giờ thực hành tuần 13**

> 💡 File → **Save a copy in Drive** trước khi sửa.

Notebook demo buổi 13 vẽ choropleth **giá** và mổ ba hình AI lỗi. Lab này bạn vẽ
tấm bản đồ **thứ hai** mà deck dặn phải có (mật độ listing — thuốc giải "bẫy diện tích"),
so phân phối bằng seaborn theo một lát cắt mới, và tự viết hồ sơ lỗi cho một hình
cherry-pick mốc thời gian.

*Lab ~50 phút; 30 phút cuối là BTL clinic (mục cuối notebook).*

## Cách làm việc trong buổi lab

- Bài tập được chia bước; mỗi bước có ô `TODO` và phần kiểm tra `assert` — chạy qua hết
  `assert` nghĩa là bạn làm đúng.
- Phần khởi động và bài có hướng dẫn: bạn nên **tự gõ, không dùng AI** — micro-exercise 🔒
  cuối giờ đo đúng các kỹ năng này.
- Bài tự làm ở cuối được gắn nhãn 🔓: bạn được dùng AI, kèm trách nhiệm khai báo
  theo chính sách AI của môn.
- Bạn kẹt quá 3 phút ở một bước: gọi trợ giảng.

## Mục tiêu

Sau buổi lab, bạn:

1. Dùng seaborn (boxplot + hue) để so phân phối giữa nhóm trên thang log.
2. Vẽ choropleth **mật độ** và ghép nó với bản đồ giá thành cặp bản đồ trung thực.
3. Xử lý quận có mặt trên bản đồ nhưng vắng trong dữ liệu (NaN sau merge geo).
4. Viết hồ sơ lỗi + bản sửa cho một hình cherry-pick mốc so sánh.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import geopandas as gpd

sns.set_theme(style="whitegrid")
BASE = "https://data.insideairbnb.com/chile/rm/santiago/2026-06-29/visualisations"
ds = pd.read_csv(f"{BASE}/listings.csv")
rv = pd.read_csv(f"{BASE}/reviews.csv", parse_dates=["date"])
len(ds), len(rv)

### Bước 1 · Boxplot + hue: giá theo phân khúc host (~12 phút)

Lát cắt của lab 5 quay lại bằng hình: giá của host chuyên nghiệp (≥5 listing) so với
host cá nhân, tách theo loại phòng — ba chiều trong một hình.

In [ ]:
ds["kieu_host"] = (ds["calculated_host_listings_count"] >= 5).map(
    {True: "chuyên nghiệp", False: "cá nhân"})
co_gia = ds.dropna(subset=["price"])

# TODO: sns.boxplot — data=co_gia, x="price", y="room_type", hue="kieu_host",
#       rồi đặt thang log cho trục x (ax.set_xscale("log")) và một title thông điệp
fig, ax = plt.subplots(figsize=(8.5, 4))
...

# --- Ô kiểm tra ---
assert ax.get_xscale() == "log", "Giá lệch phải — boxplot phải xem trên thang log"
assert len(ax.get_title()) >= 15
assert ax.get_legend() is not None, "hue phải sinh chú giải"
print("Boxplot 3 chiều đạt chuẩn.")

Đọc hình: ở **mọi** loại phòng, hộp "chuyên nghiệp" dịch nhẹ về bên phải — đúng kết
luận lab 5, giờ thấy được cả *độ chồng lấn* (hai nhóm chồng nhau nhiều — chênh lệch có
nhưng không phải hai thế giới).

### Bước 2 · Cặp bản đồ trung thực: giá + mật độ (~20 phút)

Deck cảnh báo: choropleth giá làm quận núi rộng "hét to". Thuốc giải là tấm bản đồ thứ
hai — **mật độ listing** — đặt cạnh. Ta vẽ đúng tấm đó.

In [ ]:
geo = gpd.read_file(f"{BASE}/neighbourhoods.geojson")

# TODO: tính số listing mỗi quận (groupby size, reset_index, đặt tên cột "n"),
#       merge vào geo (how="left"), đếm số quận NaN
kpi_n = ...
ban_do = ...
so_quan_nan = ...

# --- Ô kiểm tra ---
assert len(geo) == 32 and len(ban_do) == 32
assert so_quan_nan == 1
print("32 quận trên bản đồ, 31 quận có listing — 1 quận trắng dữ liệu.")

Một quận có ranh giới nhưng **không có listing nào** — geojson nhiều quận hơn dữ liệu.
Với bản đồ, NaN nên thành **0** (thật sự không có listing) và ta nói rõ điều đó trong caption
— khác với NaN "không biết" của các bài trước.

In [ ]:
# TODO: điền 0 cho n, rồi vẽ choropleth mật độ:
#       ban_do.plot(column="n", cmap="Blues", legend=True, edgecolor="#999", ax=ax)
#       + title thông điệp; tắt trục (ax.set_axis_off())
ban_do["n"] = ...
fig, ax = plt.subplots(figsize=(7, 7))
...

# --- Ô kiểm tra ---
assert ban_do["n"].isna().sum() == 0 and ban_do["n"].max() == 7182
assert not ax.axison, "Bản đồ nên tắt khung trục toạ độ"
print("Bản đồ mật độ: khối lượng dồn về cụm trung tâm nhỏ bé — ngược hẳn ấn tượng diện tích.")

Ghép tấm này cạnh bản đồ giá của notebook demo: cặp "giá + khối lượng" mới kể đủ
câu chuyện — Lo Barnechea đắt *nhưng* mỏng listing; quận Santiago rẻ *nhưng* chiếm 39%
thị trường. Một tấm choropleth đơn lẻ luôn thiếu một nửa sự thật.

### Bước 3 · Hồ sơ lỗi cho hình cherry-pick (~12 phút)

Chạy ô dưới: một hình "đúng số, sai nghĩa" kiểu AI hay đưa ra.

In [ ]:
nam = rv.set_index("date").resample("YE").size()
tu_2020 = nam.loc["2020":"2025"]

fig, ax = plt.subplots(figsize=(7, 3.2))
ax.plot(tu_2020.index.year, tu_2020.values, marker="o", color="#E62727", lw=2)
ax.set_title(f"BÙNG NỔ x{nam.loc['2025-12-31'] / nam.loc['2020-12-31']:.0f}: review tăng "
             f"{nam.loc['2025-12-31'] / nam.loc['2020-12-31']:.0f} lần kể từ 2020!")
plt.show()

In [ ]:
# Hồ sơ lỗi (điền 2 dòng trả lời vào chuỗi rồi chạy):
ho_so_loi = """
Lỗi 1 (mốc so sánh): ...
Lỗi 2 (lời so với số): ...
"""

# TODO: vẽ BẢN SỬA — cùng dữ liệu nhưng từ 2016, bỏ năm cụt 2026,
#       đánh dấu vùng COVID; title nói đúng mực
du_lieu_sua = nam.loc["2016":"2025"]
fig, ax = plt.subplots(figsize=(8, 3.2))
...

# --- Ô kiểm tra ---
assert len(ho_so_loi.strip().splitlines()) >= 2 and "..." not in ho_so_loi
assert du_lieu_sua.index.year.min() == 2016
assert len(ax.lines) >= 1 and 2016 in [int(x) for x in ax.lines[0].get_xdata()][:1] + [int(ax.lines[0].get_xdata()[0])]
print("Bản sửa cho thấy: 2020 là ĐÁY bất thường — so từ đáy thì cái gì cũng 'bùng nổ'.")

## Bài tự làm 🔓

**Tự làm 1 · plotly hover.** Dùng `plotly.express.scatter` vẽ lat/lon mẫu 5.000 listing
với `hover_name="name"`, `hover_data=["neighbourhood", "price"]` — rê chuột soi thử 5 điểm
đắt bất thường. Ghi 2 dòng: hover giúp gì mà hình tĩnh không làm được, và vì sao báo cáo
PDF vẫn cần hình tĩnh.

**Tự làm 2 · Cặp bản đồ cho thành phố của nhóm.** Vẽ cặp choropleth (trung vị giá + mật độ)
cho thành phố bài tập lớn của nhóm (mỗi snapshot Inside Airbnb đều có
`neighbourhoods.geojson`). Nhớ xử lý quận-0-listing và caption nêu n.

In [ ]:
# Viết bài tự làm của bạn ở đây

---

## 🧭 BTL clinic tuần 13 (~30 phút — theo nhóm)

Trọng tâm: **soát chất lượng hình + tiến độ hợp phần LLM.**

1. ☐ Chạy **checklist phê bình 5 câu** (deck 13) lên TỪNG hình đã có của nhóm:
   trục / đơn vị & thang / cỡ mẫu n / dạng hình / lời diễn giải. Ghi lại hình nào rớt.
2. ☐ Nếu có hình bản đồ: đã có cặp "giá + mật độ" hoặc chú thích n chưa?
3. ☐ Hợp phần LLM: ≥100 nhãn tay đã gán xong (mỗi review 2 người); accuracy
   LLM vs baseline đã đo được con số đầu tiên.
4. ☐ So sánh thành phố chính vs đối chứng: cùng thang đo / chỉ số hoá khi khác tiền tệ
   (bài "Hình C" của deck 13).
5. ☐ Kế hoạch 2 tuần cuối: ai viết mục nào của báo cáo; hạn nội bộ bản nháp
   (khuyến nghị: trước buổi 14 ba ngày).

> Nhóm xong sớm: đổi hình cho nhóm bên cạnh và phê bình chéo bằng checklist 5 câu.

## Tóm tắt buổi lab

| Bạn đã làm | Dùng cho |
|---|---|
| Boxplot + hue trên thang log | so phân phối nhiều chiều trong báo cáo |
| Cặp bản đồ giá + mật độ; NaN→0 có chủ đích | thuốc giải bẫy diện tích |
| Hồ sơ lỗi + bản sửa hình cherry-pick | phê bình hình AI (buổi 13–14) |

Buổi lý thuyết tới: **kể chuyện bằng dữ liệu & audit báo cáo AI** — buổi nội dung cuối.